In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import h5py
from utils import preprocess, calFlow3d_Wei_v1, visualization,mask,option,IO,reference
from utils import registration
import numpy as np
import nd2
import tifffile as tiff
from matplotlib import pyplot as plt
import cupy as cp

#filePath
movingFilePath="/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15849.nd2"
refenceFilePath_start='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15846.nd2'
refenceFilePath_end='/home/cyf/wbi/Virginia/f2013/250705_f2013_ubi_gcamp7f_bactin_mcherry_6dpf_15850.nd2'
config_file='/home/cyf/wbi/Virginia/wbi_0811/wholistic_registration/config.toml'
base_folder="/home/cyf/wbi/Virginia/f2013_0814registraed/"
os.makedirs(base_folder, exist_ok=True)

mem_folder = os.path.join(base_folder, "membrane")
ca_folder = os.path.join(base_folder, "ca")
concat_folder= os.path.join(base_folder, "concat")

os.makedirs(mem_folder, exist_ok=True)
os.makedirs(ca_folder, exist_ok=True)
os.makedirs(concat_folder, exist_ok=True)
#read meta data
meta_start=IO.readMeta(refenceFilePath_start)
meta_end=IO.readMeta(refenceFilePath_end)
meta_moving=IO.readMeta(movingFilePath)

#process 2000 frames in 1 iteration
total_frames = 47997
chunk_size = 5000

Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 2
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 19)
Total frames is 3
Z ratio is 3.0769230769230766
Data size is (1152, 1152, 1)
Total frames is 47997


do registration for the downsampling data

In [5]:
import os
import numpy as np
import zarr
from numcodecs import GZip

# ======================
# Global variables
# ======================
references_log = []  # store reference sources
zarrFilePath = "/home/cyf/wbi/Virginia/f2013_0912registrated.zarr"

# ======================
# Utility functions
# ======================
def compute_reference_from_block(mem_block, ca_block):
    """Generate a reference image from a block of frames"""
    mem_block = cp.asarray(mem_block)
    mem_ref, indsort = reference.pick_initial_reference(mem_block)

    ca_block = cp.asarray(ca_block)
    Ca_data_reshape = cp.reshape(ca_block, (len(mem_block), -1))
    Ca_average = cp.mean(Ca_data_reshape[indsort, :], axis=0)
    Ca_ref_1plane = cp.reshape(Ca_average, (mem_block.shape[1], mem_block.shape[2]))
    Ca_ref_1plane_transform = 250 * cp.log10(1 + Ca_ref_1plane)

    return (mem_ref + Ca_ref_1plane_transform).get()


def register_one_frame(frame_idx, mem_img, ca_img, ref_pool, ref_range):
    """Register one mem+Ca frame to the reference generated from the pool"""
    ref_img = compute_reference_from_block(
        np.array([m for m in ref_pool["mem"]]),
        np.array([c for c in ref_pool["ca"]])
    )
    mem_reg, ca_reg, _, _, _ = registration.wbi_registration_2d(
        np.expand_dims(mem_img, axis=0),
        np.expand_dims(ca_img, axis=0),
        config_file,
        ref_img
    )
    references_log.append({
        "used_for": (frame_idx, frame_idx + 1),
        "ref_from": ref_range,
        "ref_img": ref_img
    })
    return mem_reg[0], ca_reg[0], ref_img


# ======================
# Main pipeline
# ======================

# Read one frame to get image size
test_img = IO.readFrame(movingFilePath, 0, channel=0)
H, W = test_img.shape

# Downsample: take 1 frame every 10 frames
downsample = 10
num_frames_ds = total_frames // downsample

from numcodecs import Blosc
compressor = Blosc(cname="lz4", clevel=1, shuffle=Blosc.BITSHUFFLE)

# Pre-create a large Zarr file (store downsampled frames + reference)
root = zarr.open(zarrFilePath, mode="w")

mem_z = root.create_dataset(
    "membrane", shape=(num_frames_ds, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)
ca_z = root.create_dataset(
    "calcium", shape=(num_frames_ds, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)
ref_z = root.create_dataset(
    "reference", shape=(num_frames_ds, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)

# Save config
with open(config_file, "r") as f:
    root.attrs["config"] = f.read()


# ==== Step 1: process the middle block ====
chunk_size = 6000
half_chunk = chunk_size // 2
total_mid = total_frames // 2
mid_start = total_mid - half_chunk
mid_end   = mid_start + chunk_size

frames_mid = list(range(mid_start, mid_end, downsample))
mem_mid = IO.readFrame(movingFilePath, frames_mid, channel=1)
ca_mid  = IO.readFrame(movingFilePath, frames_mid, channel=0)

# reference = generated from the middle block itself
ref_img = compute_reference_from_block(mem_mid, ca_mid)
mem_mid_reg, ca_mid_reg, _, _, _ = registration.wbi_registration_2d(
    mem_mid, ca_mid, config_file, ref_img
)

mem_mid_reg = np.asarray(mem_mid_reg)
ca_mid_reg  = np.asarray(ca_mid_reg)

start_idx = mid_start // downsample
end_idx   = mid_end   // downsample
mem_z[start_idx:end_idx] = mem_mid_reg.astype("f4")
ca_z[start_idx:end_idx]  = ca_mid_reg.astype("f4")
ref_z[start_idx:end_idx] = np.repeat(ref_img[None, :, :], end_idx - start_idx, axis=0).astype("f4")

references_log.append({
    "used_for": (mid_start, mid_end),
    "ref_from": (mid_start, mid_end),
    "ref_img": ref_img
})

print(f"✅ Middle block {mid_start}~{mid_end-1} finished and set as anchor reference")


# ==== Step 2: process backward (downsampled) ====
window_size = 50
for idx in range(mid_start - downsample, -1, -downsample):
    ref_start = (idx + downsample) // downsample
    ref_end   = min((idx + window_size * downsample) // downsample, num_frames_ds)

    mem_ref_block = mem_z[ref_start:ref_end]
    ca_ref_block  = ca_z[ref_start:ref_end]

    mem_img = IO.readFrame(movingFilePath, idx, channel=1)
    ca_img  = IO.readFrame(movingFilePath, idx, channel=0)

    mem_reg, ca_reg, ref_img = register_one_frame(idx, mem_img, ca_img,
                                                  {"mem": mem_ref_block, "ca": ca_ref_block},
                                                  (ref_start * downsample, ref_end * downsample))

    z_idx = idx // downsample
    mem_z[z_idx] = mem_reg.astype("f4")
    ca_z[z_idx]  = ca_reg.astype("f4")
    ref_z[z_idx] = ref_img.astype("f4")

    if idx % 1000 == 0:
        print(f"  ↪ Processed backward up to frame {idx}")


# ==== Step 3: process forward (downsampled) ====
for idx in range(mid_end, total_frames, downsample):
    ref_start = max(0, (idx - window_size * downsample) // downsample)
    ref_end   = idx // downsample

    mem_ref_block = mem_z[ref_start:ref_end]
    ca_ref_block  = ca_z[ref_start:ref_end]

    mem_img = IO.readFrame(movingFilePath, idx, channel=1)
    ca_img  = IO.readFrame(movingFilePath, idx, channel=0)

    mem_reg, ca_reg, ref_img = register_one_frame(idx, mem_img, ca_img,
                                                  {"mem": mem_ref_block, "ca": ca_ref_block},
                                                  (ref_start * downsample, ref_end * downsample))

    z_idx = idx // downsample
    mem_z[z_idx] = mem_reg.astype("f4")
    ca_z[z_idx]  = ca_reg.astype("f4")
    ref_z[z_idx] = ref_img.astype("f4")

    if idx % 1000 == 0:
        print(f"  ↪ Processed forward up to frame {idx}")


# ==== Step 4: save log ====
np.savez(os.path.join(base_folder, "references_log_ds10.npz"), references=references_log)
print(f"Finished (1 out of every 10 frames registered), results saved to {zarrFilePath}")


Frame: 1	Initial Error is:760.0222778320312	Eventual Error: 431.7175598144531
Frame: 2	Initial Error is:759.9761352539062	Eventual Error: 413.16693115234375
Frame: 3	Initial Error is:758.00146484375	Eventual Error: 410.9184265136719
Frame: 4	Initial Error is:760.0140380859375	Eventual Error: 420.88739013671875
Frame: 5	Initial Error is:800.3453979492188	Eventual Error: 416.9283752441406
Frame: 6	Initial Error is:774.9285278320312	Eventual Error: 408.56121826171875
Frame: 7	Initial Error is:778.8265991210938	Eventual Error: 413.8848876953125
Frame: 8	Initial Error is:780.2105102539062	Eventual Error: 417.7117004394531
Frame: 9	Initial Error is:764.35400390625	Eventual Error: 411.9461669921875
Frame: 10	Initial Error is:761.8447265625	Eventual Error: 404.3446350097656
Frame: 11	Initial Error is:776.1468505859375	Eventual Error: 411.3287658691406
Frame: 12	Initial Error is:760.91650390625	Eventual Error: 402.9301452636719
Frame: 13	Initial Error is:762.5413208007812	Eventual Error: 408.25

fill the missing frames

In [ ]:
import zarr
import numpy as np
from numcodecs import GZip

# ======================
# Load downsampled results
# ======================
zarrFilePath_ds10 = "/home/cyf/wbi/Virginia/f2013_0912registrated.zarr"
root_ds10 = zarr.open(zarrFilePath_ds10, mode="r")

mem_ds10 = root_ds10["membrane"]
ca_ds10  = root_ds10["calcium"]
ref_ds10=root_ds10["reference"]

H, W = mem_ds10.shape[1:]
total_frames = IO.getTotalFrames(movingFilePath)

# ======================
# Create new Zarr file for full registration
# ======================
zarrFilePath_full = "/home/cyf/wbi/Virginia/f2013_0912registrated_full.zarr"
root_full = zarr.open(zarrFilePath_full, mode="w")

mem_full = root_full.create_dataset(
    "membrane", shape=(total_frames, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)
ca_full = root_full.create_dataset(
    "calcium", shape=(total_frames, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)
ref_full = root_full.create_dataset(
    "reference", shape=(total_frames, H, W),
    chunks=(1, H, W), dtype="f4", compressor=compressor
)

# ======================
# Helper: construct reference image
# ======================
def build_reference(mem_img, ca_img):
    return mem_img + 250 * np.log10(1 + ca_img)

# ======================
# Fill anchor (downsampled) frames
# ======================
anchors = np.arange(mem_ds10.shape[0]) * 10

# Prepare temporary arrays for anchors
mem_tmp = np.zeros((total_frames, H, W), dtype="f4")
ca_tmp  = np.zeros((total_frames, H, W), dtype="f4")
ref_tmp = np.zeros((total_frames, H, W), dtype="f4")
# Fill only anchor positions
mem_tmp[anchors] = mem_ds10[:].astype("f4")
ca_tmp[anchors]  = ca_ds10[:].astype("f4")
ref_tmp[anchors] = ref_ds10[:].astype("f4")
# Write to Zarr in one shot
mem_full[:] = mem_tmp
ca_full[:]  = ca_tmp
ref_full[:] = ref_tmp

print(f"✅ Finished writing {len(anchors)} anchor frames (batch mode)")
# ======================
# Register non-anchor frames in batches
# ======================
batch_size = 500
for start in range(0, total_frames, batch_size):
    end = min(start + batch_size, total_frames)

    # Load raw frames from original dataset
    mem_batch = IO.readFrame(movingFilePath, range(start, end), channel=1)
    ca_batch  = IO.readFrame(movingFilePath, range(start, end), channel=0)

    # Group frames by their nearest anchor index
    group = {}
    for j, idx in enumerate(range(start, end)):
        if idx % 10 == 0:  # already processed as anchor
            continue
        nearest_ds = round(idx / 10)
        if nearest_ds not in group:
            group[nearest_ds] = []
        group[nearest_ds].append((j, idx))

    # Register each group in one call (shared reference)
    for nearest_ds, items in group.items():
        ref_mem = mem_ds10[nearest_ds]
        ref_ca  = ca_ds10[nearest_ds]
        ref_img = ref_ds10[nearest_ds]

        # Select frames belonging to this group
        sel_idx = [j for j, _ in items]
        mem_sel = mem_batch[sel_idx]
        ca_sel  = ca_batch[sel_idx]

        # Register all frames in this group with the same reference
        mem_reg, ca_reg, _, _, _ = registration.wbi_registration_2d(
            mem_sel, ca_sel, config_file, ref_img
        )

        # Save results
        for k, (_, idx) in enumerate(items):
            mem_full[idx] = mem_reg[k].astype("f4")
            ca_full[idx]  = ca_reg[k].astype("f4")
            ref_full[idx] = ref_img.astype("f4")

    print(f"Processed frames {start} ~ {end-1}")

Compute reliable mask

In [ ]:
from utils import reliableAnalysis
import zarr
import numpy as np
import cupy as cp
zarrFilePath_ds10 = "/home/cyf/wbi/Virginia/f2013_0912registrated.zarr"
root_ds10 = zarr.open(zarrFilePath_ds10, mode="r")

mem_ds10 = root_ds10["membrane"][:]
ca_ds10  = root_ds10["calcium"][:]
reference_ds10=root_ds10['reference'][:]

option={
    "n": 20,
    "n2": 20,
    "n3": 400,
    "sigma": 3,
    "sigma2": 10.0,
    'decay':0.99,
    "temporalMaskThres": 3,
    "spatialMaskThres": -1
}

ca_ds10+=1
ca_ref=250*np.log10(1+ca_ds10)
temporalMask, accumulaMask, spatialMask = reliableAnalysis.get_reliable_mask(cp.asarray(mem_ds10+ca_ref) ,cp.asarray(reference_ds10), option)
